# 03 - Confidence Interval

Member: [Keymal]
Role: Inference Analyst

Research Question

1. Seberapa besar ketidakpastian estimasi probabilitas keberhasilan (misalnya PR merged)?
2. Seberapa besar ketidakpastian estimasi rata-rata kejadian (misalnya issue per hari)?
3. Apakah hasil Bayesian mendukung hasil estimasi frequentist?

Notebook ini menggunakan hasil estimasi dari Member B sebagai dasar analisis inferensi statistik.


## AI Usage Disclosure

**Member:** [Keymal] — [Inference Analyst] | **Claude:** 
| Task                          | Tool   | Prompt summary                                    | Output modified?        |
| ----------------------------- | ------ | ------------------------------------------------- | ----------------------- |
|Implementasi confidence interval | Claude | "contoh implementasi confidence interval" | Ya — menyesuaikan dengan struktur projek | 
| Debugging kode | Claude | "Periksa error dan validasi input" | Ya

Ditulis sepenuhnya tanpa AI: Seluruh interpretasi hasil analisis dan kesimpulan notebook.

# Import library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.estimator import (
    mle_bernoulli,
    beta_posterior
)

from src.inference import (
    ci_bernoulli,
    confidence_interval,
    credible_interval
)

plt.style.use("default")

Load dataset

In [ ]:
df_pulls = pd.read_csv("../data/clean/pulls_clean.csv")
df_issues = pd.read_csv("../data/clean/issues_clean.csv")

print("Jumlah Pull Requests:", len(df_pulls))
print("Jumlah Issues:", len(df_issues))

df_pulls.head()

# Confidence Interval untuk Probabilitas Pull Request Berhasil Merge

Analisis pertama menggunakan model Bernoulli untuk mengestimasi probabilitas sebuah pull request berhasil di-merge.

Variabel yang digunakan adalah:

- is_merged = 1 → Pull Request berhasil di-merge
- is_merged = 0 → Pull Request tidak di-merge

# Bernoulli MLE dan Confidence Interval

In [ ]:
k = int(df_pulls["is_merged"].sum())
n = len(df_pulls)

p_hat = mle_bernoulli(df_pulls["is_merged"])

ci_merge = ci_bernoulli(k, n)

print("Jumlah Success (k):", k)
print("Jumlah Observasi (n):", n)
print("MLE Bernoulli:", p_hat)

ci_merge

# Visualisasi Confidence Interval Bernoulli

In [ ]:
plt.figure(figsize=(8,4))

plt.axvline(
    ci_merge["lower_bound"],
    linestyle="--",
    label="Lower Bound"
)

plt.axvline(
    ci_merge["upper_bound"],
    linestyle="--",
    label="Upper Bound"
)

plt.axvline(
    p_hat,
    label="MLE"
)

plt.title("Confidence Interval Bernoulli")
plt.xlabel("Probability")
plt.legend()
plt.show()

## Interpretasi Confidence Interval Bernoulli

Berdasarkan hasil estimasi Bernoulli, probabilitas sebuah pull request berhasil di-merge adalah sebesar sekitar 87%. Nilai ini menunjukkan bahwa sebagian besar pull request yang masuk ke repository berhasil melalui proses review dan akhirnya digabungkan ke branch utama.

Confidence interval 95% digunakan untuk mengukur tingkat ketidakpastian dari estimasi tersebut. Karena jumlah observasi yang digunakan cukup besar, rentang confidence interval yang diperoleh relatif sempit sehingga estimasi yang dihasilkan dapat dianggap stabil dan representatif terhadap kondisi repository selama periode pengamatan.

Bagi maintainer proyek, hasil ini menunjukkan bahwa proses review dan integrasi kontribusi berjalan dengan tingkat keberhasilan yang tinggi.

# Confidence Interval untuk Waktu Penutupan Pull Request

Analisis kedua digunakan untuk mengestimasi rata-rata waktu yang dibutuhkan untuk menutup pull request.

# Confidence Interval Waktu Penutupan Pull Request

In [ ]:
mean_days = df_pulls["days_to_close"].mean()

std_days = df_pulls["days_to_close"].std()

n_days = len(df_pulls)

ci_days = confidence_interval(
    theta_hat=mean_days,
    sigma=std_days,
    n=n_days
)

print("Rata-rata days_to_close:", mean_days)
print("Standar Deviasi:", std_days)

ci_days

# Visualisasi distribusi Days to Close

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    df_pulls["days_to_close"],
    bins=20
)

plt.axvline(
    mean_days,
    label="Mean"
)

plt.title("Distribution of Pull Request Closing Time")
plt.xlabel("Days")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## Interpretasi Confidence Interval Waktu Penutupan Pull Request

Analisis terhadap variabel days_to_close menunjukkan bahwa rata-rata waktu yang dibutuhkan untuk menutup sebuah pull request relatif singkat. Hal ini menunjukkan bahwa sebagian besar pull request dapat ditangani dan diselesaikan dalam waktu yang cepat.

Confidence interval yang diperoleh memberikan gambaran rentang ketidakpastian terhadap estimasi rata-rata waktu penutupan tersebut. Dengan ukuran sampel yang besar, estimasi yang dihasilkan dapat dianggap cukup andal untuk merepresentasikan performa proses review repository.

# Credible Interval Menggunakan Distribusi Beta Posterior

Analisis berikut menggunakan pendekatan Bayesian dengan distribusi Beta posterior yang diperoleh dari hasil estimasi Member B.

# Beta Posterior Calculation

In [ ]:
m = n - k

posterior = beta_posterior(k, m)

posterior

# Credible Interval Calculation

In [ ]:
cred = credible_interval(
    posterior["alpha"],
    posterior["beta"]
)

cred

# Visualisasi Distribusi Beta Posterior

In [ ]:
from scipy.stats import beta

x = np.linspace(0, 1, 1000)

y = beta.pdf(
    x,
    posterior["alpha"],
    posterior["beta"]
)

plt.figure(figsize=(8,4))

plt.plot(x, y)

plt.axvline(
    cred["lower_bound"],
    linestyle="--",
    label="Lower Bound"
)

plt.axvline(
    cred["upper_bound"],
    linestyle="--",
    label="Upper Bound"
)

plt.title("Beta Posterior Distribution")
plt.xlabel("Probability")
plt.ylabel("Density")
plt.legend()
plt.show()

## Interpretasi Credible Interval

Distribusi posterior Beta dibentuk menggunakan data keberhasilan dan kegagalan pull request yang diamati pada repository.

Pendekatan Bayesian menghasilkan credible interval yang menggambarkan rentang nilai probabilitas merge yang paling konsisten dengan data historis repository. Hasil credible interval menunjukkan bahwa estimasi probabilitas merge memiliki tingkat ketidakpastian yang rendah karena didukung oleh jumlah observasi yang besar.

Konsistensi antara hasil Bayesian dan frequentist memberikan keyakinan tambahan terhadap estimasi probabilitas merge yang diperoleh.

# Summary

Berdasarkan hasil analisis inferensi statistik, probabilitas pull request berhasil di-merge berada pada tingkat yang tinggi. Selain itu, rata-rata waktu penutupan pull request menunjukkan bahwa maintainer mampu merespons kontribusi dengan cepat.

Analisis confidence interval dan credible interval menunjukkan bahwa estimasi yang diperoleh memiliki tingkat ketidakpastian yang rendah karena didukung oleh jumlah data yang cukup besar.

Secara keseluruhan, hasil inferensi statistik menunjukkan bahwa repository berada dalam kondisi yang sehat dari sisi pengelolaan kontribusi dan respons terhadap aktivitas komunitas.